# OLS Regressions for Property and Violent Crime (State/County)

This notebook contains the source code for the project's regressions. For comprehensive analysis and explanations, please refer to the **Final Report (Special Projects in Econ Research - Unemployment and Crime.pdf).**

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import scipy.stats as stats
from linearmodels import PanelOLS, RandomEffects
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

## Descriptive Stats

### State Summary Stats

In [2]:
# Import state data
state_data = pd.read_csv("Data/Special Project Research Final Data - State_long .csv")
year = pd.Categorical(state_data['year'])
state_data = state_data.set_index(["state", "year"])
state_data.head()


,,violent crime,violent_murder,violent_rape,violent_robbery,violent_assault,property crime,property_buglary,property_larceny,property_auto theft,unemployment,population,Land Area,population density,poverty rate,employee rate,per capita income,% Males 15-24 years,% Males 25-39 years
state,year,,,,,,,,,,,,,,,,,,
Alabama,2006,425.2,8.3,35.9,153.5,227.6,3936.1,969.1,2644.3,322.7,4.0,4599030.0,135767,33.874432,14.3,0.000338,31474,7.240125,9.170825
Alaska,2006,688.0,5.4,76.0,90.3,516.4,3604.9,617.3,2610.2,377.4,6.6,670053.0,1723337,0.388811,8.9,0.000879,41157,7.806415,10.701509
Arizona,2006,501.4,7.5,31.5,149.6,312.7,4627.9,925.3,2813.1,889.5,4.2,6166318.0,295234,20.886206,14.4,0.000323,34703,7.337797,10.083232
Arkansas,2006,551.6,7.3,46.5,98.4,399.4,3967.5,1139.9,2562.1,265.5,5.2,2810872.0,137732,20.408271,17.7,0.000017,29617,7.027598,9.516538
California,2006,532.5,6.8,25.3,194.7,305.7,3170.9,676.0,1829.1,665.7,4.9,36457549.0,423967,85.991478,12.2,0.000300,41746,7.821757,10.761144


In [3]:
state_cols=[0,5]
for i in range(9,18):
   state_cols.append(i)

stats_state= state_data.describe().round(2).iloc[[1,2,3,7],state_cols]

#Add row for median in the summary stats
median_state= state_data.median().to_frame().T.reset_index()
median_state["index"]="median"
median_state= median_state.set_index("index").iloc[:,state_cols].round(2)
summary_state= pd.concat([stats_state,median_state]).T

summary_state

,mean,std,min,max,median
violent crime,389.30,198.17,99.30,1508.40,343.15
property crime,2843.12,744.58,1031.90,5182.50,2765.70
unemployment,6.48,2.43,2.60,16.40,6.10
population,6064943.86,6809882.94,515004.00,39250017.00,4206074.00
Land Area,189371.35,247621.54,177.00,1723337.00,143521.50
population density,134.09,483.55,0.39,3848.42,35.42
poverty rate,13.28,3.38,5.40,23.10,12.90
employee rate,0.00,0.00,0.00,0.01,0.00
per capita income,42875.90,8001.89,27981.00,76623.00,41323.50
% Males 15-24 years,7.18,0.40,6.06,8.83,7.15


### California County Summary Stats

In [4]:
# Import County Data

ca_data = pd.read_csv("Data/California County_data.csv")
year = pd.Categorical(ca_data['year'])
ca_data = ca_data.set_index(["county", "year"])
ca_data.head(12)

violent crime  violent crime rate per 1000 persons  \
county         year                                                       
Alameda County 2012          11899                              7.70978   
               2013          11739                              7.49059   
               2014          10356                              6.51905   
               2015           9679                              6.00520   
               2016          10127                              6.21388   
               2017           9923                              6.02707   
Amador County  2012            130                              3.45276   
               2013            100                              2.68068   
               2014             81                              2.14201   
               2015             84                              2.22428   
               2016             92                              2.44246   
               2017            115                              2.99620   

                     property crime  property crime rate per 1000 persons  \
county         year                                                         
Alameda County 2012           62130                             40.256193   
               2013           60133                             38.370512   
               2014           57620                             36.271478   
               2015           60009                             37.231739   
               2016           59744                             36.658653   
               2017           63861                             38.788148   
Amador County  2012             797                             21.168097   
               2013             798                             21.391808   
               2014             629                             16.633611   
               2015             602                             15.940686   
               2016             669                             17.760905   
               2017             649                             16.908968   

                     unemployment rate  high school graduate rate  \
county         year                                                 
Alameda County 2012               0.09                       0.80   
               2013               0.07                       0.81   
               2014               0.06                       0.83   
               2015               0.05                       0.85   
               2016               0.04                       0.86   
               2017               0.04                       0.86   
Amador County  2012               0.12                       0.89   
               2013               0.10                       0.84   
               2014               0.08                       0.88   
               2015               0.07                       0.89   
               2016               0.06                       0.88   
               2017               0.05                       0.87   

                     per capita personal income population  \
county         year                                          
Alameda County 2012                       52670  1,543,365   
               2013                       54653  1,567,167   
               2014                       57951  1,588,576   
               2015                       62664  1,611,770   
               2016                       66377  1,629,738   
               2017                       70370  1,646,405   
Amador County  2012                       35944     37,651   
               2013                       36838     37,304   
               2014                       38617     37,815   
               2015                       40392     37,765   
               2016                       40934     37,667   
               2017                       41769     38,382   

                    population per 1000 land area  popu

In [5]:
ca_cols=[1,3]
for i in range(4,13):
    ca_cols.append(i)

stats_ca= ca_data.describe().round(2).iloc[[1,2,3,7],ca_cols]

#Add row for median in the summary stats
median_ca= ca_data.median().to_frame().T.reset_index()
median_ca["index"]="median"
median_ca= median_ca.set_index("index").iloc[:,ca_cols].round(2)
summary_ca= pd.concat([stats_ca,median_ca]).T
summary_ca

,mean,std,min,max,median
violent crime rate per 1000 persons,4.18,1.62,0.94,8.95,3.99
property crime rate per 1000 persons,24.29,8.26,8.14,63.22,23.20
unemployment rate,0.09,0.04,0.03,0.28,0.08
high school graduate rate,0.82,0.12,0.30,1.00,0.84
per capita personal income,47285.36,17143.52,28563.00,122748.00,41581.00
population density,707.96,2449.28,1.82,18647.49,107.76
percent of population below poverty level,16.25,5.06,4.90,28.30,16.30
full-time law enforcement employees rate,1.71,0.84,0.66,5.32,1.46
income inequality,14.29,2.68,6.82,26.77,13.94
% Males 15-24 years,7.25,1.56,3.11,11.47,7.26


## Hausman Test

In [6]:
# Define Function for Hausman Test
def perform_panel_analysis(dependent_var, independent_vars, df):
    """
    Fits Fixed Effects (Within) and Random Effects models,
    then performs a Hausman Test.
    """
    # Filter data to avoid missing values in the specific regression
    vars_needed = [dependent_var] + independent_vars
    data_clean = df[vars_needed].dropna()
    
    # Add a constant for the Random Effects model
    exog_re = sm.add_constant(data_clean[independent_vars])
    exog_fe = data_clean[independent_vars]
    endog = np.log(data_clean[dependent_var]) # Applying log to the dependent variable
    
    # 1. Fixed Effects (Within) Model
    # Time-effects can be added via time_effects=True if needed, matching R's default entity-effects
    fe_model = PanelOLS(endog, exog_fe, entity_effects=True).fit()
    
    # 2. Random Effects Model
    re_model = RandomEffects(endog, exog_re).fit()
    
    # 3. Hausman Test Calculation
    # Extract coefficients (excluding the constant from RE to match dimensions)
    b_fe = fe_model.params
    b_re = re_model.params.drop('const', errors='ignore')
    
    # Extract covariance matrices
    v_fe = fe_model.cov
    v_re = re_model.cov.drop(index='const', columns='const', errors='ignore')
    
    # Calculate Chi-squared statistic and p-value
    coef_diff = b_fe - b_re
    cov_diff = v_fe - v_re
    
    try:
        chi2_stat = np.dot(np.dot(coef_diff.T, np.linalg.inv(cov_diff)), coef_diff)
        df_deg = len(b_fe)
        p_val = 1 - stats.chi2.cdf(chi2_stat, df_deg)
    except np.linalg.LinAlgError:
        chi2_stat, p_val = np.nan, np.nan
        print(f"Warning: Matrix inversion failed for {dependent_var}. Check for multicollinearity.")

    print(f"=== Results for log({dependent_var}) ===")
    print(f"Hausman Test Chi2: {chi2_stat:.4f}, p-value: {p_val:.4f}")
    if p_val < 0.05:
        print("Conclusion: Reject H0. Use Fixed Effects (Within) model.")
    else:
        print("Conclusion: Fail to reject H0. Random Effects model is preferred.")
        
    return fe_model, re_model

### Hausman Test for States

In [7]:
# Independent variables list
X_vars = ['unemployment', 'population','population density','poverty rate', 'employee rate','per capita income', '% Males 15-24 years', '% Males 25-39 years']

# Run analysis for all categories
crimes = ['violent crime', 'property crime']

# Dictionary to store model outputs 
models_fe = {}
models_re = {}

for crime in crimes:
    try:
        fe_mod, re_mod = perform_panel_analysis(crime, X_vars, state_data)
        models_fe[crime] = fe_mod
        models_re[crime] = re_mod
        print("\n" + "-"*50 + "\n")
    except Exception as e:
        print(f"Could not process {crime}: {e}")

=== Results for log(violent crime) ===
Hausman Test Chi2: 20.7589, p-value: 0.0078
Conclusion: Reject H0. Use Fixed Effects (Within) model.

--------------------------------------------------

=== Results for log(property crime) ===
Hausman Test Chi2: 43.1450, p-value: 0.0000
Conclusion: Reject H0. Use Fixed Effects (Within) model.

--------------------------------------------------



### Hausman Test for California County

In [8]:
# Independent variables list

X_vars = ['unemployment rate','high school graduate rate','per capita personal income','percent of population below poverty level','full-time law enforcement employees rate','income inequality','% Males 15-24 years','% Males 25-39 years','population density'] 

# Run analysis for all categories
crimes = ['violent crime rate per 1000 persons', 'property crime rate per 1000 persons']

# Dictionary to store model outputs 
models_fe = {}
models_re = {}

for crime in crimes:
    try:
        fe_mod, re_mod = perform_panel_analysis(crime, X_vars, ca_data)
        models_fe[crime] = fe_mod
        models_re[crime] = re_mod
        print("\n" + "-"*50 + "\n")
    except Exception as e:
        print(f"Could not process {crime}: {e}")

=== Results for log(violent crime rate per 1000 persons) ===
Hausman Test Chi2: 86.0515, p-value: 0.0000
Conclusion: Reject H0. Use Fixed Effects (Within) model.

--------------------------------------------------

=== Results for log(property crime rate per 1000 persons) ===
Hausman Test Chi2: 37.3314, p-value: 0.0000
Conclusion: Reject H0. Use Fixed Effects (Within) model.

--------------------------------------------------



## OLS Regressions

### State Violent Crime OLS Regression

In [9]:
exog_var = ['unemployment', 'population density', 'poverty rate', 'employee rate',
            'per capita income', '% Males 15-24 years', '% Males 25-39 years']
exog = sm.add_constant(state_data[exog_var])

# Run OLS Regression and set entity_effects=True for Fixed effects
mod = PanelOLS(state_data['violent crime'], exog, weights = None, entity_effects=True) 
res = mod.fit()
print(res)

                          PanelOLS Estimation Summary                           
Dep. Variable:          violent crime   R-squared:                        0.3394
Estimator:                   PanelOLS   R-squared (Between):             -1.6401
No. Observations:                 572   R-squared (Within):               0.3394
Date:                Tue, May 19 2026   R-squared (Overall):             -1.5351
Time:                        20:31:32   Log-likelihood                   -2878.1
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      37.655
Entities:                          52   P-value                           0.0000
Avg Obs:                       11.000   Distribution:                   F(7,513)
Min Obs:                       11.000                                           
Max Obs:                       11.000   F-statistic (robust):             37.655
                            

### State Property Crime OLS Regression

In [10]:
exog_var = ['unemployment', 'population density', 'poverty rate', 'employee rate',
            'per capita income', '% Males 15-24 years', '% Males 25-39 years']
exog = sm.add_constant(state_data[exog_var])

# Run OLS Regression and set entity_effects=True for Fixed effects
mod = PanelOLS(state_data['property crime'], exog, weights = None, entity_effects=True)
res = mod.fit()
print(res)

                          PanelOLS Estimation Summary                           
Dep. Variable:         property crime   R-squared:                        0.5221
Estimator:                   PanelOLS   R-squared (Between):              0.1807
No. Observations:                 572   R-squared (Within):               0.5221
Date:                Tue, May 19 2026   R-squared (Overall):              0.2426
Time:                        20:31:32   Log-likelihood                   -3893.9
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      80.078
Entities:                          52   P-value                           0.0000
Avg Obs:                       11.000   Distribution:                   F(7,513)
Min Obs:                       11.000                                           
Max Obs:                       11.000   F-statistic (robust):             80.078
                            

### County Violent Crime OLS Regression

In [11]:
exog_var = ['unemployment rate','high school graduate rate','per capita personal income','percent of population below poverty level','full-time law enforcement employees rate','income inequality','% Males 15-24 years','% Males 25-39 years','population density'] 
exog = sm.add_constant(ca_data[exog_var])

# Run OLS Regression and set entity_effects=True for Fixed effects
mod = PanelOLS(ca_data['violent crime rate per 1000 persons'], exog, weights = None, entity_effects=True)
res = mod.fit()
print(res)

                                   PanelOLS Estimation Summary                                   
Dep. Variable:     violent crime rate per 1000 persons   R-squared:                        0.1757
Estimator:                                    PanelOLS   R-squared (Between):             -2.1765
No. Observations:                                  342   R-squared (Within):               0.1757
Date:                                 Tue, May 19 2026   R-squared (Overall):             -1.8182
Time:                                         20:31:32   Log-likelihood                   -295.18
Cov. Estimator:                             Unadjusted                                           
                                                         F-statistic:                      6.5346
Entities:                                           57   P-value                           0.0000
Avg Obs:                                        6.0000   Distribution:                   F(9,276)
Min Obs:            

#### County Property Crime OLS Regression

In [12]:
exog_var = ['unemployment rate','high school graduate rate','per capita personal income','percent of population below poverty level','full-time law enforcement employees rate','income inequality','% Males 15-24 years','% Males 25-39 years','population density'] 
exog = sm.add_constant(ca_data[exog_var])

# Run OLS Regression and set entity_effects=True for Fixed effects
mod = PanelOLS(ca_data['property crime rate per 1000 persons'], exog, weights = None, entity_effects=True)
res = mod.fit()
print(res)


                                   PanelOLS Estimation Summary                                    
Dep. Variable:     property crime rate per 1000 persons   R-squared:                        0.2830
Estimator:                                     PanelOLS   R-squared (Between):             -11.246
No. Observations:                                   342   R-squared (Within):               0.2830
Date:                                  Tue, May 19 2026   R-squared (Overall):             -10.109
Time:                                          20:31:32   Log-likelihood                   -753.64
Cov. Estimator:                              Unadjusted                                           
                                                          F-statistic:                      12.106
Entities:                                            57   P-value                           0.0000
Avg Obs:                                         6.0000   Distribution:                   F(9,276)
Min Obs:  